# W4D1 — Convolution by Hand — Lab

**Week 4 · Day 1 · CNNs & Model Fine-Tuning** · Lab

Nine multiplications and a sum. That is the whole operation, and by the end of this notebook you
will have written it twice: once by hand on the lecture's 5×5 grid, printing the slide's `−3`,
and once as a function that runs over a 640×480 photograph and makes its edges appear.

Three things, in order:

- **The slide's numbers, reproduced exactly.** The top-left position must print `−3`, and the full
  output must be the `−3 0 +3` grid. Not approximately — the arithmetic is integer arithmetic.
- **Your own `conv2d`, with stride and padding.** Written as two loops, because the loops are the
  mechanism. You will predict every output shape with a formula and then check it against the array
  you actually got.
- **A real photograph.** The same nine multiplications, run over a street scene. The edges appear.
  Nobody told the filter where they were.

<div dir="rtl" align="right">

# الأسبوع ٤ · اليوم ١ — الالتفاف باليد

**الأسبوع الرابع · اليوم الأول · الشبكات الالتفافية وضبط النماذج** · معمل

تسع عمليات ضرب وجمعها. هذه هي العملية كلها، وستكتبها في هذا الدفتر مرّتين: مرّة بيدك على شبكة
المحاضرة ٥×٥ فتطبع `−٣` كما في الشريحة، ومرّة كدالة تمرّ على صورة ٦٤٠×٤٨٠ فتُظهر حوافها.

ثلاثة أشياء بالترتيب:

- **أرقام الشريحة كما هي.** يجب أن يطبع الموضع الأول `−٣`، وأن يكون الخرج كاملًا شبكة `−٣ ٠ +٣`.
  لا تقريبًا — فالحساب هنا حساب أعداد صحيحة.
- **دالة `conv2d` من كتابتك، بخطوة وحاشية.** مكتوبة بحلقتين، لأن الحلقتين هما الآلية نفسها.
  ستتنبّأ بكل حجم خرج بالمعادلة ثم تتحقّق منه بالمصفوفة التي حصلت عليها فعلًا.
- **صورة حقيقية.** العمليات التسع نفسها على مشهد شارع، فتظهر الحواف. ولم يخبر أحدٌ المرشِّح أين هي.

</div>

> **This is your lab notebook.** Work through the hints — they tell you what to do and where
> to look, not what to type. Stuck for more than ten minutes on one task? Open the `_guided`
> version. That is not cheating; sitting stuck in silence is the only mistake. The full
> solution is released at the end of the day.

<div dir="rtl" align="right">

> **هذا دفتر المعمل الخاص بك.** اعمل وفق الإرشادات — فهي تخبرك بما يجب فعله وأين تبحث، لا بما
> تكتبه حرفيًا. إذا توقّفت أكثر من عشر دقائق عند مهمة واحدة فافتح نسخة `_guided`؛ هذا ليس غشًّا،
> والخطأ الوحيد هو أن تبقى متوقّفًا بصمت. ويُنشر الحل الكامل في نهاية اليوم.

</div>

## Learning objectives

By the end of this lab you can:

- Compute one output position of a convolution by hand, writing out all nine products, and get the
  lecture's `−3`.
- Write `conv2d(image, kernel, padding, stride)` with two loops, and say what each loop indexes.
- Predict an output size with `(n − f + 2p) // s + 1` and confirm it against the array's real shape.
- Apply a hand-written kernel to a photograph and name what the result responded to.
- Say why a colour filter has 27 weights and not 9, and how much slower your loops are than a
  library that does the same arithmetic.

<div dir="rtl" align="right">

## أهداف التعلّم

في نهاية هذا المعمل تستطيع:

- أن تحسب موضعًا واحدًا من الالتفاف بيدك، وتكتب حواصل الضرب التسعة كلها، فتحصل على `−٣` كالمحاضرة.
- أن تكتب `conv2d(image, kernel, padding, stride)` بحلقتين، وأن تقول ما الذي تفهرسه كل حلقة.
- أن تتنبّأ بحجم الخرج بالمعادلة `(n − f + 2p) // s + 1` وتتحقّق منه بالشكل الحقيقي للمصفوفة.
- أن تُطبّق مرشِّحًا كتبته بيدك على صورة وتسمّي ما الذي استجاب له الناتج.
- أن تقول لماذا يملك مرشِّح الصور الملوّنة ٢٧ وزنًا لا ٩، وكم حلقاتك أبطأ من مكتبة تفعل الحساب نفسه.

</div>

## About the data

**Dataset:** `sample_photos` — 20 photographs, longest side 640 px, ~1.8 MB, downloaded once and
cached.

They are the Attribution-licensed (CC BY 2.0 / CC BY-SA 2.0) subset of COCO train2017: ordinary
street scenes, each carrying at least two of *person*, *car*, *stop sign*. `ATTRIBUTION.csv` inside
the archive credits every photograph.

One row here is one JPEG. There is no target column — today the picture *is* the data, and the
thing you are predicting is nothing at all. You are transforming.

**The known problem with them:** they are JPEGs. JPEG compression works by throwing away
high-frequency detail, which is exactly what an edge is. Zoom into any edge map you produce today
and you will find faint 8×8 blocks that no edge in the world put there — they are the compression
grid. It does not spoil the lab, but it is the reason a real vision pipeline that cares about fine
detail does not start from a JPEG.

Two of the twenty are deliberately hard, and `HARD_IMAGES.txt` names them. They matter on **D3**,
not today: an object detector finds 0% and 33% of their labelled objects. Today they are just
photographs like the rest.

<div dir="rtl" align="right">

## عن البيانات

**مجموعة البيانات:** `sample_photos` — عشرون صورة، ضلعها الأطول ٦٤٠ بكسل، نحو ١٫٨ ميغابايت، تُنزَّل
مرّة وتُخزَّن.

وهي المجموعة الجزئية المرخَّصة بالنسب (CC BY 2.0 / CC BY-SA 2.0) من COCO train2017: مشاهد شوارع
عادية، في كل واحدة فئتان على الأقل من: شخص، سيارة، لوحة قف. وملف `ATTRIBUTION.csv` داخل الأرشيف
ينسب كل صورة إلى مصدرها.

الصف الواحد هنا صورة JPEG واحدة. ولا يوجد عمود هدف — فاليوم الصورة **هي** البيانات، والذي تتنبّأ به
لا شيء. أنت تُحوِّل لا تتنبّأ.

**المشكلة المعروفة فيها:** أنها صور JPEG. وضغط JPEG يعمل بإسقاط التفاصيل عالية التردّد، وهي بالضبط
ما تكونه الحافة. قرّب النظر في أي خريطة حواف تُنتجها اليوم فستجد مربّعات باهتة ٨×٨ لم تضعها أي حافة
في الواقع — إنها شبكة الضغط. وهذا لا يُفسد المعمل، لكنه سبب أن خط الرؤية الحقيقي الذي يهتمّ
بالتفاصيل الدقيقة لا يبدأ من JPEG.

صورتان من العشرين صعبتان عمدًا، وملف `HARD_IMAGES.txt` يسمّيهما. وأهميتهما في **اليوم الثالث** لا
اليوم: إذ يجد كاشف الكائنات صفر٪ و٣٣٪ من كائناتهما المُعلَّمة. أما اليوم فهما صورتان كغيرهما.

</div>

## Setup

Run this first. It fetches the photographs, unpacks them once, and fixes the seed.

<div dir="rtl" align="right">

## الإعداد

شغّل هذه الخلية أولًا. تجلب الصور وتفكّ ضغطها مرّة واحدة وتثبّت البذرة العشوائية.

</div>

In [ ]:
# === AIEP portable setup — works locally (conda) and on Google Colab ===============
try:
    import aiep
except ImportError:
    import subprocess, sys
    from pathlib import Path
    # A clone that never ran `pip install -e shared/` still has the package on disk —
    # use it before reaching for the network. Colab (no clone) falls through to pip.
    _local = next((p / "shared" for p in [Path.cwd(), *Path.cwd().parents]
                   if (p / "shared" / "aiep").is_dir()), None)
    if _local:
        sys.path.insert(0, str(_local))
    else:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                               "git+https://github.com/0xRush/AIEP_Olo_student.git#subdirectory=shared"])
    import aiep

from aiep.env import ensure, seed_everything, device, versions
from aiep.data import get_dataset_dir, describe_dataset
from aiep.paths import ARTEFACT_DIR
from aiep.checks import check, check_close, check_shape, report

ensure("scipy", "pillow")                # ← only this lab's extra packages
seed_everything(42)                      # course-wide seed

import numpy as np
from PIL import Image

PHOTOS = get_dataset_dir("sample_photos") / "photos"
PHOTO_FILES = sorted(PHOTOS.glob("*.jpg"))
EDGES = ARTEFACT_DIR / "edges"
EDGES.mkdir(parents=True, exist_ok=True)

print(describe_dataset("sample_photos"))
print(f"\n{len(PHOTO_FILES)} photographs in {PHOTOS}")
print(versions(), "| device:", device())

## Section 1 — Warm-up: the slide's nine products  (≈25 min)

Everything in this section already works. Run it, then change one number and watch what moves.

The image from the morning is 5×5, zeros everywhere except a vertical bar down the middle column.
The filter is 3×3: `+1` on the left, `0` in the middle, `−1` on the right. Read it out loud as a
question — *"is it brighter on my left than on my right?"* — because that is all it asks, at every
position it can reach.

<div dir="rtl" align="right">

## القسم الأول — الإحماء: حواصل الضرب التسعة من الشريحة (نحو ٢٥ دقيقة)

كل ما في هذا القسم يعمل أصلًا. شغّله ثم غيّر رقمًا واحدًا وراقب ما الذي يتحرّك.

صورة الصباح ٥×٥، أصفار في كل مكان إلا عمودًا رأسيًا في المنتصف. والمرشِّح ٣×٣: `+١` على اليسار
و`٠` في الوسط و`−١` على اليمين. اقرأه بصوتٍ عالٍ كسؤال — *«هل الجهة التي على يساري أنصع من التي على
يميني؟»* — فهذا كل ما يسأله، عند كل موضع يستطيع بلوغه.

</div>

In [ ]:
IMAGE = np.zeros((5, 5), dtype=int)
IMAGE[:, 2] = 1                      # the vertical bar, down the middle column

KERNEL = np.array([[1, 0, -1],
                   [1, 0, -1],
                   [1, 0, -1]])

print("image:\n", IMAGE)
print("\nkernel:\n", KERNEL)
print("\nthe filter asks one question at every position: brighter on the left than the right?")

### The top-left position, all nine products written out

Lay the filter over the top-left 3×3 corner of the image. Multiply the nine pairs, add them.

The cell prints each product on its own line, then the sum. **It must print `-3`** — the number on
the slide.

<div dir="rtl" align="right">

### الموضع الأول أعلى اليسار، بحواصل الضرب التسعة كلها

ضع المرشِّح فوق الزاوية العليا اليسرى ٣×٣ من الصورة. اضرب الأزواج التسعة ثم اجمعها.

تطبع الخلية كل حاصل ضرب في سطر ثم المجموع. **ويجب أن تطبع `-3`** — وهو رقم الشريحة.

</div>

In [ ]:
window = IMAGE[0:3, 0:3]             # the top-left corner the filter covers first
print("window:\n", window, "\n")

total = 0
for row in range(3):
    for col in range(3):
        product = window[row, col] * KERNEL[row, col]
        total += product
        print(f"  image[{row},{col}]={window[row, col]}  x  kernel[{row},{col}]={KERNEL[row, col]:>2}"
              f"  =  {product:>2}")

print(f"\nsum of the nine products = {total}")
assert total == -3, "the slide says -3; if this is not -3 the arithmetic is wrong"

### Why negative?

The bar sits in column 2. Over the top-left window the bar is on the **right**, under the `−1`
column of the filter, and there is nothing under the `+1` column. Bright on the right, dark on the
left — so the filter answers with a large negative number.

That sign is information, not noise: **`−3` marks the bar's left edge** (dark, then bright), and
`+3` at the other end marks its right edge. A student who reports `|−3|` has thrown away which way
the brightness went.

**Change one thing:** set `IMAGE[:, 2] = 1` to `IMAGE[:, 1] = 1` in the cell above, re-run both
cells, and watch the `−3` move to a different output column. The filter did not move. The edge did.

<div dir="rtl" align="right">

### لماذا كان سالبًا؟

العمود المضيء في الفهرس ٢. وفي النافذة العليا اليسرى يقع هذا العمود على **اليمين**، أي تحت عمود
`−١` من المرشِّح، ولا شيء تحت عمود `+١`. فالنصوع على اليمين والعتمة على اليسار، فيجيب المرشِّح بعدد
سالب كبير.

والإشارة معلومة لا ضجيج: **`−٣` تُعلّم الحافة اليسرى للعمود** (عتمة ثم نصوع)، و`+٣` في الطرف الآخر
تُعلّم حافته اليمنى. ومن يعرض `|−٣|` يكون قد رمى اتّجاه تغيّر النصوع.

**غيّر شيئًا واحدًا:** اجعل `IMAGE[:, 2] = 1` في الخلية أعلاه `IMAGE[:, 1] = 1`، وأعد تشغيل
الخليتين، وراقب انتقال `−٣` إلى عمود خرج آخر. لم يتحرّك المرشِّح، بل تحرّكت الحافة.

</div>

## Section 2 — Core: six tasks  (≈60 min)

1. `conv2d(image, kernel)` — two loops, stride 1, no padding.
2. Run it on the 5×5 and match `scipy.signal.correlate2d` to the last decimal.
3. `output_size(n, f, p, s)` — the formula, checked against three counted answers.
4. Padding and stride inside `conv2d`, and the real shapes checked against your predictions.
5. A real photograph, vertical then horizontal. **The edges appear.**
6. Three more kernels: blur, sharpen, and one you design yourself.

<div dir="rtl" align="right">

## القسم الثاني — الأساسي: ست مهام (نحو ٦٠ دقيقة)

١. `conv2d(image, kernel)` — حلقتان، خطوة ١، بلا حاشية.
٢. شغّلها على الشبكة ٥×٥ وطابِق `scipy.signal.correlate2d` حتى آخر منزلة.
٣. `output_size(n, f, p, s)` — المعادلة، متحقَّقًا منها بثلاث إجابات معدودة باليد.
٤. الحاشية والخطوة داخل `conv2d`، والأشكال الحقيقية متحقَّقًا منها بتنبّؤاتك.
٥. صورة حقيقية، رأسيًا ثم أفقيًا. **فتظهر الحواف.**
٦. ثلاثة مرشِّحات أخرى: تمويه، وتحديد، وواحد من تصميمك.

</div>

### Task 2.1 — `conv2d`, with two loops

Write the function. Two loops: the outer one walks down the output rows, the inner one across the
output columns. At each position, slice the window out of the image, multiply it elementwise by the
kernel, and sum.

**Do not vectorise it.** There is a one-line version using stride tricks and it is 85× faster; you
will meet a faster version in the stretch section and `nn.Conv2d` on Wednesday. Today the loops
*are* the lesson — the indices you write here are the ones the fast version hides.

Return a float array, not an integer one: the blur kernel in task 2.6 has fractional weights.

<div dir="rtl" align="right">

### المهمة ٢٫١ — `conv2d` بحلقتين

اكتب الدالة. حلقتان: الخارجية تمشي على صفوف الخرج، والداخلية على أعمدته. وعند كل موضع اقتطع النافذة
من الصورة واضربها عنصرًا بعنصر في المرشِّح ثم اجمع.

**لا تكتبها بصيغة متّجهية.** هناك صيغة من سطر واحد بحيل الخطوات وهي أسرع خمسة وثمانين ضعفًا؛ وستلقى صيغة
أسرع في القسم الإضافي، وتلقى `nn.Conv2d` يوم الأربعاء. أمّا اليوم فالحلقتان **هما** الدرس — فالفهارس
التي تكتبها هنا هي التي تُخفيها الصيغة السريعة.

وأرجِع مصفوفة أعداد عشرية لا صحيحة: فمرشِّح التمويه في المهمة ٢٫٦ أوزانه كسرية.

</div>

In [ ]:
# ────────────────────────────────────────────────────────────────────
# 1) The output is smaller than the input. Work out its height and width first:
#    how many positions does a 3-tall kernel fit into a 5-tall image? Count them.
# 2) Two loops over those output positions. At (i, j) the window starts at row i,
#    column j and is as tall and wide as the kernel.
# 3) One position is a slice, an elementwise multiply, and a sum — you already
#    wrote it in the warm-up, with the loops spelled out.
# 4) Return float, so a kernel of 1/9s does not get rounded to zeros.
# Search: "numpy slice window elementwise multiply sum"
# https://numpy.org/doc/stable/user/basics.indexing.html
#
# ١) الخرج أصغر من الدخل. احسب ارتفاعه وعرضه أولًا: كم موضعًا يتّسع له مرشِّح
#    ارتفاعه ٣ داخل صورة ارتفاعها ٥؟ عُدّها.
# ٢) حلقتان على مواضع الخرج تلك. وعند (i, j) تبدأ النافذة من الصف i والعمود j
#    وبارتفاع المرشِّح وعرضه.
# ٣) الموضع الواحد اقتطاع وضرب عنصريّ وجمع — وقد كتبته في الإحماء
#    بحلقتيه مكتوبتين صراحةً.
# ٤) أرجِع أعدادًا عشرية، لئلا يُقرَّب مرشِّح أوزانه ١/٩ إلى أصفار.
# ابحث عن: "numpy slice window elementwise multiply sum"
# https://numpy.org/doc/stable/user/basics.indexing.html
# ────────────────────────────────────────────────────────────────────

    # TODO: window at each position, multiply by the kernel, sum. Return floats.
    # مهمة: واضربها في المرشِّح واجمع. وأرجِع أعدادًا عشرية.

### Task 2.2 — match the slide, then match the library

Two checks on the same function, and they are checks of different things.

The first is against the slide: the output must be the `−3 0 +3` grid, on all three rows, exactly.
These are integers; `np.array_equal` is the right test and `np.isclose` would be hiding something.

The second is against `scipy.signal.correlate2d(..., mode="valid")`, which does the same arithmetic
in C. If your function disagrees with it anywhere, your function is wrong — and the most common way
to be wrong is to have written *convolution* proper, which flips the kernel first. Everything
called "convolution" in deep learning is in fact cross-correlation, unflipped. The name is a
historical accident that costs one debugging session per person.

<div dir="rtl" align="right">

### المهمة ٢٫٢ — طابِق الشريحة ثم طابِق المكتبة

فحصان على الدالة نفسها، وهما فحصان لشيئين مختلفين.

الأول مقابل الشريحة: يجب أن يكون الخرج شبكة `−٣ ٠ +٣` في صفوفها الثلاثة تمامًا. وهذه أعداد صحيحة،
فـ`np.array_equal` هو الفحص الصحيح، و`np.isclose` كان سيُخفي شيئًا.

والثاني مقابل `scipy.signal.correlate2d(..., mode="valid")` التي تُجري الحساب نفسه بلغة C. فإن
اختلفت دالتك عنها في أي موضع فدالتك خاطئة — وأشيع صور الخطأ أن تكون قد كتبت **الالتفاف** بمعناه
الرياضي الذي يقلب المرشِّح أولًا. فكل ما يُسمّى «التفافًا» في التعلّم العميق هو في الحقيقة ارتباط
متبادل بلا قلب. والاسم حادثة تاريخية تكلّف كل شخص جلسة تنقيح واحدة.

</div>

In [ ]:
# ────────────────────────────────────────────────────────────────────
# 1) Build the expected 3x3 grid literally — three rows of (-3, 0, 3) — and compare
#    it to your output with an exact array comparison, not a tolerance.
# 2) Import correlate2d from scipy.signal and run it with mode="valid", which is
#    the no-padding case you implemented.
# 3) Compare the two arrays to 1e-9 and print the largest disagreement, so a
#    failure tells you how wrong you are rather than just that you are.
# Search: "scipy signal correlate2d valid mode"
# https://docs.scipy.org/doc/scipy/reference/generated/scipy.signal.correlate2d.html
#
# ١) ابنِ الشبكة المتوقَّعة ٣×٣ صراحةً — ثلاثة صفوف من (−٣، ٠، ٣) — وقارنها بخرجك
#    بمقارنة مصفوفات تامّة لا بتسامح عددي.
# ٢) استورد `correlate2d` من `scipy.signal` وشغّلها بـ`mode="valid"` وهي حالة
#    «بلا حاشية» التي نفّذتها.
# ٣) قارن المصفوفتين حتى ١e−٩ واطبع أكبر خلاف، ليخبرك الفشل بمقدار
#    خطئك لا بوقوعه فقط.
# ابحث عن: "scipy signal correlate2d valid mode"
# https://docs.scipy.org/doc/scipy/reference/generated/scipy.signal.correlate2d.html
# ────────────────────────────────────────────────────────────────────

from scipy.signal import correlate2d
                         [-3, 0, 3],
                         [-3, 0, 3]])

### Task 2.3 — output size, counted and then computed

Before you write the formula, count. A 3-wide filter on a 5-wide image starts at column 0, then 1,
then 2 — and at column 3 it would hang off the edge. Three positions. Three output columns.

Now write `output_size(n, f, p, s)` for `(n − f + 2p) // s + 1` and check it against three answers
you can count on your fingers:

| n | f | p | s | counted |
|---|---|---|---|---|
| 5 | 3 | 0 | 1 | 3 |
| 5 | 3 | 1 | 1 | 5 |
| 5 | 3 | 0 | 2 | 2 |

Use integer division. With a stride that does not divide evenly the filter simply stops early —
`//` is the arithmetic of "how many whole steps fit", and a float answer would be a size no array
can have.

<div dir="rtl" align="right">

### المهمة ٢٫٣ — حجم الخرج، عدًّا ثم حسابًا

قبل أن تكتب المعادلة، عُدّ. مرشِّح عرضه ٣ على صورة عرضها ٥ يبدأ من العمود ٠ ثم ١ ثم ٢، وعند العمود ٣
يخرج عن الحافة. ثلاثة مواضع. ثلاثة أعمدة خرج.

اكتب الآن `output_size(n, f, p, s)` للمعادلة `(n − f + 2p) // s + 1` وتحقّق منها بثلاث إجابات
تعدّها على أصابعك:

| n | f | p | s | بالعدّ |
|---|---|---|---|---|
| ٥ | ٣ | ٠ | ١ | ٣ |
| ٥ | ٣ | ١ | ١ | ٥ |
| ٥ | ٣ | ٠ | ٢ | ٢ |

واستخدم القسمة الصحيحة. فإذا لم تقسم الخطوةُ الطولَ قسمةً تامّة توقّف المرشِّح مبكرًا — و`//` هو حساب
«كم خطوة كاملة تتّسع»، والإجابة العشرية حجمٌ لا تملكه أي مصفوفة.

</div>

In [ ]:
# ────────────────────────────────────────────────────────────────────
# 1) One line: the formula from the slide, with integer division.
# 2) The three cases are (5,3,0,1) -> 3, (5,3,1,1) -> 5, (5,3,0,2) -> 2. Loop over
#    them and print predicted against counted, so a wrong formula is visible.
# 3) Assert all three before moving on — task 2.4 checks real arrays against this.
# Search: "convolution output size formula stride padding"
# https://pytorch.org/docs/stable/generated/torch.nn.Conv2d.html
#
# ١) سطر واحد: معادلة الشريحة بقسمة صحيحة.
# ٢) الحالات الثلاث: (٥،٣،٠،١) ← ٣، و(٥،٣،١،١) ← ٥، و(٥،٣،٠،٢) ← ٢. كرّر عليها
#    واطبع المتوقَّع مقابل المعدود، لتظهر المعادلة الخاطئة.
# ٣) تحقّق من الثلاث قبل المتابعة — فالمهمة ٢٫٤ تفحص مصفوفات حقيقية مقابلها.
# ابحث عن: "convolution output size formula stride padding"
# https://pytorch.org/docs/stable/generated/torch.nn.Conv2d.html
# ────────────────────────────────────────────────────────────────────

COUNTED = [(5, 3, 0, 1, 3), (5, 3, 1, 1, 5), (5, 3, 0, 2, 2)]

### Task 2.4 — padding and stride, and the shapes checked against the formula

Extend `conv2d` with `padding` and `stride`. Padding first: `np.pad` a ring of zeros around the
image before the loops start, and everything downstream is unchanged. Stride second: it changes
where each window *starts*, not how big it is.

Then the point of the task. For each of the three cases from 2.3, run the real function on the real
5×5 image and compare `result.shape` against what `output_size` predicted. A formula you have
checked against an array is knowledge; a formula you believe is a liability at 3 a.m. on Wednesday
when a model refuses to build.

<div dir="rtl" align="right">

### المهمة ٢٫٤ — الحاشية والخطوة، والأشكال متحقَّقًا منها بالمعادلة

وسّع `conv2d` بـ`padding` و`stride`. الحاشية أولًا: أضِف بـ`np.pad` حلقة أصفار حول الصورة قبل بدء
الحلقتين، فلا يتغيّر شيء بعدها. ثم الخطوة: وهي تغيّر **من أين تبدأ** كل نافذة لا كم حجمها.

ثم يأتي مقصد المهمة. لكل حالة من حالات ٢٫٣ الثلاث، شغّل الدالة الحقيقية على الصورة ٥×٥ الحقيقية
وقارن `result.shape` بما تنبّأت به `output_size`. فالمعادلة التي فحصتها بمصفوفة معرفة، والمعادلة
التي تصدّقها فقط عبءٌ يوم الأربعاء الثالثة فجرًا حين يرفض النموذج أن يُبنى.

</div>

In [ ]:
# ────────────────────────────────────────────────────────────────────
# 1) Pad first, with np.pad and a constant of 0, then run exactly the loops you
#    already wrote — padding is a change to the input, not to the algorithm.
# 2) Stride changes where a window starts: output position i reads image row i*s.
#    The output height is what output_size() says, not what it was at stride 1.
# 3) Check all three (p, s) cases against output_size and print both numbers.
# Search: "numpy pad constant values zero"
# https://numpy.org/doc/stable/reference/generated/numpy.pad.html
#
# ١) أضِف الحاشية أولًا بـ`np.pad` بثابت صفر، ثم شغّل الحلقتين اللتين كتبتهما — فالحاشية
#    تغييرٌ في الدخل لا في الخوارزمية.
# ٢) تغيّر الخطوةُ بدايةَ النافذة: فموضع الخرج i يقرأ صف الصورة i×s. وارتفاع الخرج هو
#    ما تقوله `output_size` لا ما كان عند الخطوة ١.
# ٣) افحص حالات (p, s) الثلاث كلها مقابل `output_size` واطبع الرقمين.
# ابحث عن: "numpy pad constant values zero"
# https://numpy.org/doc/stable/reference/generated/numpy.pad.html
# ────────────────────────────────────────────────────────────────────

    # TODO: (i, j) now starts at row i*stride, column j*stride.
    # مهمة: (i, j) تبدأ الآن من الصف i×stride والعمود j×stride.
SHAPES = {}
for n, f, p, s, counted in COUNTED:
    # TODO: record the real shape so the sanity check can compare them.
    # مهمة: الحقيقي ليقارن بينهما فحص السلامة.

### Task 2.5 — a real photograph, and the edges appear

Load one of the twenty photographs, convert it to greyscale, and run your `conv2d` over it with the
vertical-edge kernel. Then the horizontal one — the same three numbers, turned on their side:

```
 1  1  1
 0  0  0
-1 -1 -1
```

Save both results as PNGs into `edges/`. To save a signed result as an image you have to decide
what to do with the sign; take the absolute value and scale the maximum to 255. Say to yourself
what that discards — the direction of every edge, which is exactly what the `−3` versus `+3`
distinction was in the warm-up.

Look at the two images side by side. The vertical filter finds the sides of buildings, poles and
door frames; the horizontal one finds the tops of cars, the road and the skyline. **Nobody told
either filter what to look for.** They are nine numbers each.

<div dir="rtl" align="right">

### المهمة ٢٫٥ — صورة حقيقية، فتظهر الحواف

حمّل صورة من العشرين، وحوّلها إلى تدرّج رمادي، وشغّل `conv2d` عليها بمرشِّح الحواف الرأسية. ثم
بالمرشِّح الأفقي — وهو الأعداد الثلاثة نفسها مقلوبةً على جنبها:

```
 1  1  1
 0  0  0
-1 -1 -1
```

واحفظ الناتجين صورتَي PNG في `edges/`. ولحفظ ناتج ذي إشارة كصورة عليك أن تقرّر ما تفعله بالإشارة؛
خُذ القيمة المطلقة واجعل أكبر قيمة ٢٥٥. وقل لنفسك ما الذي يُهدره ذلك — اتّجاه كل حافة، وهو بعينه
الفرق بين `−٣` و`+٣` في الإحماء.

وانظر إلى الصورتين جنبًا إلى جنب. يجد المرشِّح الرأسي جوانب المباني والأعمدة وإطارات الأبواب، ويجد
الأفقي أسطح السيارات والطريق وخط الأفق. **ولم يخبر أحدٌ أيًّا منهما بما يبحث عنه.** كلٌّ منهما تسعة
أعداد.

</div>

In [ ]:
# ────────────────────────────────────────────────────────────────────
# 1) Open the file with PIL, .convert("L") for greyscale, then np.asarray(..., float).
# 2) The horizontal kernel is the vertical one transposed — one numpy attribute.
# 3) Run conv2d twice. On a 480x640 photo the loops take a few seconds each; that
#    slowness is the stretch section's subject, so time it now if you are curious.
# 4) To save: absolute value, divide by the max, multiply by 255, cast to uint8,
#    and hand that to Image.fromarray.
# Search: "PIL Image fromarray uint8 grayscale save"
# https://pillow.readthedocs.io/en/stable/reference/Image.html
#
# ١) افتح الملف بـPIL ثم `.convert("L")` للتدرّج الرمادي ثم `np.asarray(..., float)`.
# ٢) المرشِّح الأفقي هو الرأسي منقولًا — سمة واحدة في numpy.
# ٣) شغّل `conv2d` مرّتين. وعلى صورة ٤٨٠×٦٤٠ تستغرق الحلقتان ثوانيَ في كل مرّة؛ وهذا
#    البطء موضوع القسم الإضافي، فقِسه الآن إن شئت.
# ٤) للحفظ: القيمة المطلقة، ثم القسمة على الأكبر، ثم الضرب في ٢٥٥، ثم `uint8`،
#    وسلّمها إلى `Image.fromarray`.
# ابحث عن: "PIL Image fromarray uint8 grayscale save"
# https://pillow.readthedocs.io/en/stable/reference/Image.html
# ────────────────────────────────────────────────────────────────────

import matplotlib.pyplot as plt
PHOTO = PHOTO_FILES[0]
    # TODO: PNG into EDGES and return its path.
    # مهمة: في `EDGES` وأرجِع مسارها.
# TODO: Convolve the greyscale photo with the vertical kernel, then with its transpose.
# مهمة: التفّ الصورة الرمادية بالمرشِّح الرأسي ثم بمنقوله.
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (title, data) in zip(axes, [("original", grey),
    ax.imshow(data, cmap="gray")
    ax.set_title(title)
    ax.axis("off")
plt.tight_layout()
plt.show()

### Task 2.6 — three more kernels, and what each one responded to

Three kernels, then a sentence each.

- **Blur** — a 3×3 of `1/9`. Every output pixel is the average of its neighbourhood.
- **Sharpen** — `[[0,−1,0],[−1,5,−1],[0,−1,0]]`. Read it: five parts *this* pixel, minus one part
  of each neighbour. Where the neighbourhood already agrees, the weights cancel to 1 and nothing
  happens; where it disagrees, the difference is amplified.
- **A diagonal detector you design.** Same idea as the vertical kernel, rotated 45°. Two minutes
  with a pen beats ten minutes of guessing at the keyboard.

Save all three to `edges/` and fill in `RESPONDED_TO` with one sentence each — what the bright
parts of that output actually are, in that photograph. The dictionary is checked at the end, and
the TA reads what you wrote.

<div dir="rtl" align="right">

### المهمة ٢٫٦ — ثلاثة مرشِّحات أخرى، وما الذي استجاب له كلٌّ منها

ثلاثة مرشِّحات ثم جملة عن كلٍّ منها.

- **التمويه** — شبكة ٣×٣ من `١/٩`. فكل بكسل خرج متوسّط جيرانه.
- **التحديد** — `[[0,−1,0],[−1,5,−1],[0,−1,0]]`. اقرأه: خمسة أجزاء من **هذا** البكسل ناقص جزء من كل
  جار. فحيث يتّفق الجوار تتلاشى الأوزان إلى ١ ولا يحدث شيء، وحيث يختلف يُضخَّم الفرق.
- **كاشف قطريّ من تصميمك.** الفكرة نفسها كالمرشِّح الرأسي مُدارةً ٤٥ درجة. ودقيقتان بقلم خير من عشر
  دقائق تخمينًا على لوحة المفاتيح.

احفظ الثلاثة في `edges/` واملأ `RESPONDED_TO` بجملة عن كلٍّ منها — ما هي الأجزاء المضيئة من ذلك
الخرج فعلًا في تلك الصورة. والقاموس مفحوص في النهاية، والمساعد يقرأ ما كتبت.

</div>

In [ ]:
# ────────────────────────────────────────────────────────────────────
# 1) Blur is np.ones((3,3)) / 9. Sharpen is the array printed in the task text.
# 2) For the diagonal, put the +1s along one diagonal and the -1s along the other,
#    with zeros between — the vertical kernel's logic, turned 45 degrees.
# 3) Run each through conv2d, save each with save_map, and look at all three before
#    you write a word. The sentence is about this photograph, not about kernels.
# 4) A blurred image should look blurred. If yours is dark, you scaled it wrong.
# Search: "image kernel blur sharpen convolution 3x3"
# https://en.wikipedia.org/wiki/Kernel_(image_processing)
#
# ١) التمويه `np.ones((3,3)) / 9`. والتحديد هو المصفوفة المطبوعة في نص المهمة.
# ٢) وللقطريّ ضع `+١` على أحد القطرين و`−١` على الآخر وأصفارًا بينهما — منطق المرشِّح
#    الرأسي مُدارًا ٤٥ درجة.
# ٣) شغّل كلًّا منها بـ`conv2d` واحفظها بـ`save_map`، وانظر إلى الثلاثة قبل أن تكتب
#    كلمة. فالجملة عن هذه الصورة لا عن المرشِّحات عمومًا.
# ٤) الصورة المموّهة يجب أن تبدو مموّهة. فإن خرجت عندك معتمة فقد قِستها خطأ.
# ابحث عن: "image kernel blur sharpen convolution 3x3"
# https://en.wikipedia.org/wiki/Kernel_(image_processing)
# ────────────────────────────────────────────────────────────────────

# TODO: a diagonal detector you design yourself.
# مهمة: وكاشفًا قطريًّا من تصميمك أنت.
    # TODO: One sentence per kernel: what are the bright parts of that output, in this photo?
    # مهمة: جملة لكل مرشِّح: ما هي الأجزاء المضيئة من ذلك الخرج في هذه الصورة؟

## Section 3 — Stretch: colour, and the price of two loops  (≈30 min)

Two open tasks. The first is the one that trips people in week 6; the second is why PyTorch exists.

<div dir="rtl" align="right">

## القسم الثالث — الإضافي: الألوان وثمن الحلقتين (نحو ٣٠ دقيقة)

مهمّتان مفتوحتان. الأولى هي التي تُعثِر الناس في الأسبوع السادس، والثانية هي سبب وجود PyTorch.

</div>

### Stretch A — a colour filter has 27 weights

A colour photograph is not a 480×640 grid. It is 480×640×**3** — red, green and blue, stacked.
A 3×3 filter over it is therefore 3×3×3 = **27 numbers**, not 9, and it produces **one** output
map, not three: the three channel responses are summed before they leave the filter.

That summing is the part people miss. A filter does not process the channels separately and keep
them apart; it collapses them. Which is why the *output* depth of a conv layer is the number of
filters you asked for, and has nothing to do with the input depth.

Write it, run it on a colour photograph, and assert the weight count is 27.

<div dir="rtl" align="right">

### الإضافي أ — مرشِّح الصورة الملوّنة له ٢٧ وزنًا

الصورة الملوّنة ليست شبكة ٤٨٠×٦٤٠، بل ٤٨٠×٦٤٠×**٣**: أحمر وأخضر وأزرق مكدَّسة. ولذلك يكون المرشِّح
٣×٣ عليها ٣×٣×٣ = **٢٧ عددًا** لا ٩، ويُنتج خريطة خرج **واحدة** لا ثلاثًا: إذ تُجمع استجابات القنوات
الثلاث قبل أن تخرج من المرشِّح.

وهذا الجمع هو ما يفوت الناس. فالمرشِّح لا يعالج القنوات منفصلةً ويُبقيها منفصلة، بل يطويها. ولهذا
يكون **عمق خرج** الطبقة الالتفافية هو عدد المرشِّحات التي طلبتها، ولا علاقة له بعمق الدخل.

اكتبها وشغّلها على صورة ملوّنة وتحقّق أن عدد الأوزان ٢٧.

</div>

In [ ]:
# ────────────────────────────────────────────────────────────────────
# 1) Load the photo without .convert("L") — you want its three channels this time.
# 2) The kernel now has shape (3, 3, 3). Stack your 2-D kernel three times, or build
#    three different ones and see what changes when the channels disagree.
# 3) The window is image[i:i+3, j:j+3, :] — all three channels at that position.
#    Multiply, then sum over everything, including the channel axis.
# 4) kernel.size is the weight count. It has to be 27.
# Search: "numpy sum over all axes elementwise 3d window"
# https://numpy.org/doc/stable/reference/generated/numpy.sum.html
#
# ١) حمّل الصورة بلا `.convert("L")` — فأنت تريد قنواتها الثلاث هذه المرّة.
# ٢) شكل المرشِّح الآن (٣، ٣، ٣). كدّس مرشِّحك ثنائي البعد ثلاث مرّات، أو ابنِ ثلاثة
#    مختلفة وانظر ما الذي يتغيّر حين تختلف القنوات.
# ٣) النافذة `image[i:i+3, j:j+3, :]` أي القنوات الثلاث عند ذلك الموضع. اضرب ثم اجمع
#    على كل المحاور بما فيها محور القناة.
# ٤) `kernel.size` هو عدد الأوزان. ويجب أن يكون ٢٧.
# ابحث عن: "numpy sum over all axes elementwise 3d window"
# https://numpy.org/doc/stable/reference/generated/numpy.sum.html
# ────────────────────────────────────────────────────────────────────

    # TODO: sum runs over the channel axis too — one filter, one output map.
    # مهمة: محور القناة أيضًا — مرشِّح واحد وخريطة خرج واحدة.
colour = np.asarray(Image.open(PHOTO).convert("RGB"), dtype=float)

### Stretch B — how much your loops cost

Time `conv2d` against `scipy.signal.correlate2d` on the same photograph and the same kernel. They
compute the identical answer — you proved that in task 2.2 — so the ratio is pure overhead:
Python's interpreter walking 300,000 positions one at a time, against compiled code that walks them
in a tight C loop.

On the machine this notebook was written on the ratio was about **85×**. Yours will differ; report
what you measure.

Then hold the number next to Wednesday. A trained CNN's first layer runs 32 filters over a batch of
images, thousands of times per epoch. At 85× the difference between a framework and your loops is
the difference between four minutes and five hours — which is why nobody writes the loops twice,
and why it was worth writing them once.

<div dir="rtl" align="right">

### الإضافي ب — كم تكلّفك حلقتاك

قِس زمن `conv2d` مقابل `scipy.signal.correlate2d` على الصورة نفسها والمرشِّح نفسه. فهما يحسبان
الإجابة نفسها — وقد أثبتّ ذلك في المهمة ٢٫٢ — فالنسبة إذن كلفة زائدة خالصة: مُفسِّر بايثون يمشي على
ثلاثمئة ألف موضع واحدًا واحدًا، مقابل شيفرة مُصرَّفة تمشي عليها في حلقة C ضيّقة.

وعلى الجهاز الذي كُتب عليه هذا الدفتر كانت النسبة نحو **٨٥×**. وستختلف عندك؛ فاعرض ما تقيسه أنت.

ثم ضع الرقم إلى جانب يوم الأربعاء. فالطبقة الأولى في شبكة التفافية مُدرَّبة تُمرِّر ٣٢ مرشِّحًا على
دفعة صور آلاف المرّات في الحقبة الواحدة. وعند ٨٥× يكون الفرق بين إطار عمل وحلقاتك هو الفرق بين أربع
دقائق وخمس ساعات — ولهذا لا يكتب أحد الحلقتين مرّتين، ولهذا استحقّ أن تكتبهما مرّة.

</div>

In [ ]:
# ────────────────────────────────────────────────────────────────────
# 1) time.perf_counter() around each call — not time.time(), which is coarser.
# 2) Time the two on the same array and the same kernel, or the ratio means nothing.
# 3) Report the ratio, and check the answers still agree — a fast wrong answer is
#    the failure mode this comparison is meant to catch.
# Search: "python time perf_counter benchmark function"
# https://docs.python.org/3/library/time.html#time.perf_counter
#
# ١) `time.perf_counter()` حول كل نداء — لا `time.time()` فهي أخشن.
# ٢) قِس الاثنين على المصفوفة نفسها والمرشِّح نفسه، وإلا فالنسبة بلا معنى.
# ٣) اعرض النسبة وتحقّق أن الإجابتين ما زالتا متّفقتين — فالإجابة السريعة الخاطئة هي
#    نمط الفشل الذي وُضعت هذه المقارنة لتصطاده.
# ابحث عن: "python time perf_counter benchmark function"
# https://docs.python.org/3/library/time.html#time.perf_counter
# ────────────────────────────────────────────────────────────────────

import time
# TODO: in SPEED_RATIO along with both durations.
# مهمة: `SPEED_RATIO` مع المدّتين.

## Save your artefact

`conv_check.json` holds the three things tomorrow can check against: the slide's output grid, the
three output sizes, and the speed ratio you measured. It is small on purpose — the PNGs stay in
`edges/` and are not part of the chain.

<div dir="rtl" align="right">

## احفظ أثرك

يحفظ `conv_check.json` ثلاثة أشياء يستطيع الغد التحقّق منها: شبكة خرج الشريحة، وأحجام الخرج الثلاثة،
ونسبة السرعة التي قِستها. وهو صغير عمدًا — فصور PNG تبقى في `edges/` وليست جزءًا من السلسلة.

</div>

In [ ]:
import json

conv_check = {
    "slide_output": mine.astype(int).tolist(),
    "top_left": int(mine[0, 0]),
    "output_sizes": {f"n{n}_f{f}_p{p}_s{s}": output_size(n, f, p, s)
                     for n, f, p, s, _ in COUNTED},
    "real_shapes": {f"p{p}_s{s}": list(shape) for (p, s), shape in SHAPES.items()},
    "colour_kernel_weights": int(COLOUR_KERNEL.size),
    "loop_vs_scipy_ratio": round(float(SPEED_RATIO), 1),
    "photo": PHOTO.name,
    "kernels_saved": sorted(p.name for p in EDGES.glob("*.png")),
}

path = ARTEFACT_DIR / "conv_check.json"
path.write_text(json.dumps(conv_check, indent=2))
print(f"wrote {path}")
print(json.dumps({k: v for k, v in conv_check.items() if k != "real_shapes"}, indent=2)[:400])

## Sanity check

<div dir="rtl" align="right">

## فحص سلامة

</div>

In [ ]:
check(int(mine[0, 0]) == -3,
      f"the top-left output must be exactly -3, as on the slide — got {int(mine[0, 0])}",
      f"يجب أن يكون خرج أعلى اليسار `-3` تمامًا كما في الشريحة، والناتج {int(mine[0, 0])}")

check(np.array_equal(mine.astype(int), SLIDE_OUTPUT),
      f"the full output must be the -3 0 +3 grid — got {mine.astype(int).tolist()}",
      f"يجب أن يكون الخرج كاملًا شبكة `-3 0 +3`، والناتج {mine.astype(int).tolist()}")

check(np.abs(mine - theirs).max() < 1e-9,
      f"your conv2d must match scipy's correlate2d to 1e-9 — largest gap "
      f"{np.abs(mine - theirs).max():.2e} (a flipped kernel is the usual cause)",
      f"يجب أن تطابق `conv2d` عندك دالة `correlate2d` حتى ١e-٩، وأكبر فرق "
      f"{np.abs(mine - theirs).max():.2e} (والسبب المعتاد قلب المرشِّح)")

check(all(SHAPES[(p, s)] == (output_size(n, f, p, s), output_size(n, f, p, s))
          for n, f, p, s, _ in COUNTED),
      f"every real output shape must equal what output_size predicted — got {SHAPES}",
      f"يجب أن يساوي كل شكل خرج حقيقي ما تنبّأت به `output_size`، والنتيجة {SHAPES}")

check(COLOUR_KERNEL.size == 27,
      f"a 3x3 filter on a colour image holds 27 weights, not 9 — got {COLOUR_KERNEL.size}",
      f"المرشِّح ٣×٣ على صورة ملوّنة يحمل ٢٧ وزنًا لا ٩، والناتج {COLOUR_KERNEL.size}")

saved = sorted(EDGES.glob("*.png"))
sizes = [Image.open(p).size for p in saved]
check(len(saved) >= 5 and all(min(s) >= 200 for s in sizes),
      f"at least five edge maps of 200px or more must be saved in edges/ — found "
      f"{len(saved)}: {[p.name for p in saved]}",
      f"يجب حفظ خمس خرائط حواف على الأقل بحجم ٢٠٠ بكسل فأكثر في `edges/`، والموجود "
      f"{len(saved)}: {[p.name for p in saved]}")

check(all(len(RESPONDED_TO.get(k, "").split()) >= 8 for k in ("blur", "sharpen", "diagonal")),
      f"write a real sentence for each of the three kernels — got "
      f"{ {k: len(v.split()) for k, v in RESPONDED_TO.items()} } words",
      f"اكتب جملة حقيقية لكل مرشِّح من الثلاثة، وعدد الكلمات الحالي "
      f"{ {k: len(v.split()) for k, v in RESPONDED_TO.items()} }")

report()

## What's next

Tomorrow you stop choosing the numbers. **W4D2 — CNN on MNIST**: the same operation, but the nine
weights are learned by gradient descent instead of written down, thirty-two filters at a time. Your
first task is to max-pool the `−3 0 +3` grid you produced today, so keep this notebook's output
where you can find it.

<div dir="rtl" align="right">

## ما التالي

غدًا تتوقّف عن اختيار الأعداد. **الأسبوع ٤ اليوم ٢ — شبكة التفافية على MNIST**: العملية نفسها، لكن
الأوزان التسعة تتعلّمها الشبكة بالنزول الاشتقاقي بدل أن تكتبها أنت، اثنين وثلاثين مرشِّحًا في المرّة.
ومهمّتك الأولى فيه أن تُطبّق التجميع الأقصى على شبكة `−٣ ٠ +٣` التي أنتجتها اليوم، فأبقِ خرج هذا
الدفتر في متناولك.

</div>